# Hodgkin-Huxley: the squid giant axon

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/10_hodgkin_huxley.ipynb)

The four-variable Hodgkin-Huxley model (v in mV, t in ms, i in µA/cm²). At `i = 10` it
fires repeatedly with a period of about 14.64 ms. The phase plane here is a projection
(v, n): nullclines and the direction field need all four variables, so they are off.

*Based on an example from [XPP Studio](https://xpp-studio.github.io/xpp-studio/) (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.2 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.3.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("hodgkin_huxley")
      .function("am(v)", "if(abs(v+40)<1e-7)then(1)else(0.1*(v+40)/(1-exp(-(v+40)/10)))")
      .function("bm(v)", "4*exp(-(v+65)/18)")
      .function("ah(v)", "0.07*exp(-(v+65)/20)")
      .function("bh(v)", "1/(1+exp(-(v+35)/10))")
      .function("an(v)", "if(abs(v+55)<1e-7)then(0.1)else(0.01*(v+55)/(1-exp(-(v+55)/10)))")
      .function("bn(v)", "0.125*exp(-(v+65)/80)")
      .ode("v", "(i-gna*m^3*h*(v-ena)-gk*n^4*(v-ek)-gl*(v-el))/cm")
      .ode("m", "am(v)*(1-m)-bm(v)*m")
      .ode("h", "ah(v)*(1-h)-bh(v)*h")
      .ode("n", "an(v)*(1-n)-bn(v)*n")
      .parameter(i=10, gna=120, gk=36, gl=0.3, ena=50, ek=-77, el=-54.4, cm=1)
      .initial(v=-65, m=0.05, h=0.6, n=0.32)
      .option(total=300, dt=0.02, meth="rk4"))
print(mb.render())

am(v)=if(abs(v+40)<1e-7)then(1)else(0.1*(v+40)/(1-exp(-(v+40)/10)))
bm(v)=4*exp(-(v+65)/18)
ah(v)=0.07*exp(-(v+65)/20)
bh(v)=1/(1+exp(-(v+35)/10))
an(v)=if(abs(v+55)<1e-7)then(0.1)else(0.01*(v+55)/(1-exp(-(v+55)/10)))
bn(v)=0.125*exp(-(v+65)/80)
v'=(i-gna*m^3*h*(v-ena)-gk*n^4*(v-ek)-gl*(v-el))/cm
m'=am(v)*(1-m)-bm(v)*m
h'=ah(v)*(1-h)-bh(v)*h
n'=an(v)*(1-n)-bn(v)*n
par i=10,gna=120,gk=36,gl=0.3,ena=50,ek=-77,el=-54.4,cm=1
init v=-65,m=0.05,h=0.6,n=0.32
@ total=300,dt=0.02,meth=rungekutta
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="series", x="v", y="n", window=(-85, 55, 0.2, 0.85),
              sliders={"i": (0, 30), "gk": (20, 50)}, show=["v", "n", "h"],
              auto="i", auto_range=(0, 200), auto2="gk", auto2_range=(20, 50), title="Hodgkin-Huxley")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**The period at `i = 10`** with XPP 8.1's Initialconds > Periodic, which finds one
period of the stable orbit in double precision.

In [5]:
m = ex.m
m.run(i=10, total=300, trans=0)
m.buttons.initialconds.periodic()
print(m.transcript.strip().splitlines()[-1])
print("period:", round(m.numerics["total"], 4), "ms")

message Period T=14.638325. Starting at max of V; total set to T
period: 14.6383 ms


**Hopf points and the fold of orbits** (the bistable window between them).

In [6]:
ex.set(i=0)                     # start on the rest state at i = 0, below both Hopf points
parts = ex.run_auto()
hb = sorted(p.x for d in parts for p in d.special("HB"))
print("Hopf points in i:", [round(h, 4) for h in hb])
ex.pick("HB", hb[0]); po, l1 = ex.periodic()
if po is not None:
    print("first Hopf: l1 =", round(l1, 3), "; folds of orbits:", sorted({round(p.x, 4) for p in po.special("LP")}))

Hopf points in i: [9.7792, 154.5267]


first Hopf: l1 = 0.015 ; folds of orbits: [6.2642, 7.8462, 7.9217, 154.5263]


## Things to try

- Lower `i` to 7: depending on the start the cell rests or fires (bistability between the fold at 6.26 and the Hopf at 9.78).